In [1]:
import os

import dotenv
dotenv.load_dotenv()
from kbel.disambiguators import Disambiguator
from kbel.core.mention import Mention
from kbel.core.mention import EntityType
from kbel.core.kbel import KBEL
# import logging
# logging.basicConfig(level=logging.DEBUG)

### Creating a KBEL instance (without disambiguator)

In [2]:
kbel = KBEL('wikidata')

#### Getting candidates to a mention:

In [3]:
results = kbel.candidates_lookup(Mention(label='rock', text='', entity_type=EntityType.ITEM), limit=5)
display (*results)

Candidate(iri='http://www.wikidata.org/entity/Q11399', label='rock music', id='http://www.wikidata.org/entity/Q11399', description='popular music genre', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q22731', label='stone', id='http://www.wikidata.org/entity/Q22731', description='rock; building material', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q21003018', label='Rock', id='http://www.wikidata.org/entity/Q21003018', description='family name', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q1851974', label='Rock', id='http://www.wikidata.org/entity/Q1851974', description='village in Worcestershire, England, United Kingdom', aliases=[], metadata={})

Candidate(iri='http://www.wikidata.org/entity/Q1404150', label='rock', id='http://www.wikidata.org/entity/Q1404150', description='mass of stone projecting out of the ground or water', aliases=[], metadata={})

### Using `naive` disambiguator to link entities from Wikidata

In [4]:
from kbel.disambiguators.naive import NaiveDisambiguator
kbel.disambiguator = Disambiguator('naive')

results = kbel.link(
    mention=Mention(label='rock', text='', entity_type=EntityType.ITEM))
display (*results)

('Rock', 'family name', Item(IRI('http://www.wikidata.org/entity/Q21003018')))

### Using `similarity` disambiguator to link entities from Wikidata

In [5]:
# pip install "kbel[similarity]"
from kbel.disambiguators.similarity import SimilarityDisambiguator
kbel.disambiguator = Disambiguator('sim')


/home/marcelo/Documents/projects/kbel/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1268.27it/s]


In [6]:
results = kbel.link(
    mention=Mention(label='Rock', text='Rock is a stone', entity_type=EntityType.ITEM),
    limit=2)
display (*results)

('stone',
 'rock; building material',
 Item(IRI('http://www.wikidata.org/entity/Q22731')))

('Rock',
 'male given name',
 Item(IRI('http://www.wikidata.org/entity/Q60589667')))

##### Linking a property instead of an Item

In [7]:
results = kbel.link(
    mention=Mention(label='instance of', text='Rock is a stone', entity_type=EntityType.PROPERTY), limit=1
)

display(*results)

('instance of',
 'type to which this subject corresponds/belongs. Different from P279 (subclass of); for example: K2 is an instance of mountain; volcano is a subclass of mountain',
 Property(IRI('http://www.wikidata.org/entity/P31'), None))

### Using `LLM` disambiguator to link entities from Wikidata

Instantiating LLM Disambiguator with OpenAI's models

In [9]:
# pip install "kbel[llm]"
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-5.2', api_key=os.environ['LLM_API_KEY'])

from kbel.disambiguators.llm import LLM_Disambiguator
kbel.disambiguator = Disambiguator('llm', model= model)
kbel.knowledge_base='wikidata'

In [10]:
results = kbel.link(
    mention=Mention(label='Rock', text='A rock can be used in construction to mimic the appearance and durability of natural stone.', 
                    entity_type=EntityType.ITEM), limit=2)

display (*results)

('rock',
 'naturally occurring solid aggregate of one or more minerals or mineraloids',
 Item(IRI('http://www.wikidata.org/entity/Q8063')))

##### Adding context to improve disambiguation:

In [11]:
results = kbel.link(
    Mention(
        label="Python",
        text="Python is used for coding.",
        entity_type=EntityType.ITEM,
        context="""
        Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
        """
    )
)

display(*results)

('Python',
 'general-purpose programming language',
 Item(IRI('http://www.wikidata.org/entity/Q28865')))

### Changing the knowledge source, e.g., DBpedia

In [12]:
kbel.knowledge_base = 'dbpedia'
results = kbel.link(
    Mention(
        label="Python",
        text="Python is used for coding.",
        entity_type=EntityType.ITEM,
        context="""
        Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
        """
    )
)

display(*results)

('Python (programming language)',
 '',
 Item(IRI('http://dbpedia.org/resource/Python_(programming_language)')))